# 🧠 Train Vietnamese COOLANT Model

**Production training notebook for Vietnamese fake news detection**

This notebook trains the COOLANT model on Vietnamese multimodal data using:

-   ResNet50 image features (2048-dim)
-   PhoBERT text features (768-dim)
-   HDF5-based memory-efficient data loading
-   Multi-task training (Similarity, CLIP, Detection)
-   MLflow experiment tracking

📂 Input: `notebooks/processed_data/hdf5/vifactcheck_*.h5`
💾 Output: `training/checkpoints/` + MLflow artifacts


In [ ]:
# Setup and imports
import sys
import os
import math
import random
import json
import warnings
import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
import mlflow
import mlflow.pytorch
from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# Project path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"✅ Project root: {project_root}")

In [ ]:
# Device and seeds
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"🔧 Device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Paths
HDF5_DIR = project_root / "notebooks" / "processed_data" / "hdf5"
SAVE_DIR = project_root / "training" / "checkpoints"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 HDF5 dir: {HDF5_DIR}")
print(f"💾 Save dir: {SAVE_DIR}")

In [ ]:
# Load data using HDF5DatasetSimple
from src.processing.hdf5_dataset import create_hdf5_dataloaders_from_files

BATCH_SIZE = 32

train_path = HDF5_DIR / "vifactcheck_train.h5"
dev_path = HDF5_DIR / "vifactcheck_dev.h5"
test_path = HDF5_DIR / "vifactcheck_test.h5"

print("Loading data...")
loaders, datasets = create_hdf5_dataloaders_from_files(
    train_path=str(train_path),
    dev_path=str(dev_path),
    test_path=str(test_path),
    batch_size=BATCH_SIZE,
    num_workers=0,
)

# Verify shapes
sample_text, sample_image, sample_label = datasets["train"][0]
print(f"\n📏 Sample shapes:")
print(f"  Text: {sample_text.shape} (expected: [768, 512])")
print(f"  Image: {sample_image.shape} (expected: [2048])")
print(f"  Label: {sample_label}")
print(f"\n✅ All DataLoaders ready!")

In [ ]:
# Import and setup ResNetCOOLANT model
from src.models.resnet_coolant import (
    ResNetCOOLANT,
    patch_encoding,
    patch_clip_projection,
    patch_cnn_with_dropout,
)
import models.coolant_official as coolant
from models.base import FastCNN

# Dimensions
IMAGE_DIM = 2048
TEXT_EMBED = 768
TEXT_SEQ_LEN = 512

# Config
CONFIG = {
    "shared_dim": 128,
    "sim_dim": 64,
    "clip_embed_dim": 64,
    "feature_dim": 96,
    "h_dim": 64,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "max_epochs": 30,
    "patience": 5,
    "dropout": 0.3,
    "label_smoothing": 0.1,
    "grad_clip": 1.0,
    "seed": SEED,
    "device": str(DEVICE),
    "batch_size": BATCH_SIZE,
}

print("✅ Config defined")

In [ ]:
# Create and patch model
model = ResNetCOOLANT(CONFIG)

# Patch encodings for 2048-dim image features
patch_encoding(model.similarity_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.ambiguity_module.encoding, image_dim=IMAGE_DIM)

# Patch CLIP projections
patch_clip_projection(model.clip_module, target_dim=IMAGE_DIM, is_image=True)
patch_clip_projection(model.clip_module, target_dim=TEXT_EMBED, is_image=False)

# Patch FastCNN with dropout
patch_cnn_with_dropout(
    model.similarity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"]
)
patch_cnn_with_dropout(
    model.detection_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"]
)
patch_cnn_with_dropout(
    model.detection_module.ambiguity_module.encoding.shared_text_encoding,
    TEXT_EMBED,
    CONFIG["dropout"],
)

# Move to device
model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model ready with {n_params:,} trainable parameters")

In [ ]:
# make_pair_sim function
def make_pair_sim(text, image, label, hard_negative_ratio=0.3):
    """
    Create paired/unpaired samples for similarity learning.
    """
    batch_size = text.size(0)
    device = text.device

    n_anchors = max(2, batch_size // 2)
    anchor_indices = torch.randperm(batch_size)[:n_anchors]

    t_anchor = text[anchor_indices].clone()
    i_anchor = image[anchor_indices].clone()

    i_positive = i_anchor.clone()

    n_hard = int(n_anchors * hard_negative_ratio)
    n_random = n_anchors - n_hard

    i_negative = i_anchor.clone()
    if n_hard > 0:
        i_negative[:n_hard] = i_anchor[:n_hard].roll(1, dims=0)

    if n_random > 0:
        offset = random.randint(2, max(3, n_anchors // 2))
        i_negative[n_hard:] = i_anchor[n_hard:].roll(offset, dims=0)

    return t_anchor, i_positive, i_negative


print("✅ make_pair_sim defined")

In [ ]:
# make_pair_sim function
def make_pair_sim(text, image, label, hard_negative_ratio=0.3):
    """
    Create paired/unpaired samples for similarity learning.

    Following the original COOLANT paper: use only label=0 (real) samples
    for constructing matched/unmatched pairs.
    """
    batch_size = text.size(0)
    device = text.device

    # Filter for real (label=0) samples, following the original COOLANT paper
    real_indices = (label == 0).nonzero(as_tuple=True)[0]

    if len(real_indices) < 2:
        # Fallback: use all samples if not enough real samples
        real_indices = torch.arange(batch_size, device=device)

    n_anchors = max(2, len(real_indices) // 2)
    anchor_indices = real_indices[torch.randperm(len(real_indices))[:n_anchors]]

    t_anchor = text[anchor_indices].clone()
    i_anchor = image[anchor_indices].clone()

    i_positive = i_anchor.clone()

    n_hard = int(n_anchors * hard_negative_ratio)
    n_random = n_anchors - n_hard

    i_negative = i_anchor.clone()
    if n_hard > 0:
        i_negative[:n_hard] = i_anchor[:n_hard].roll(1, dims=0)

    if n_random > 0:
        offset = random.randint(2, max(3, n_anchors // 2))
        i_negative[n_hard:] = i_anchor[n_hard:].roll(offset, dims=0)

    return t_anchor, i_positive, i_negative


print("✅ make_pair_sim defined (paper-faithful: uses label=0 samples only)")

In [ ]:
# Cosine annealing scheduler with warmup
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.current_step = 0
        self.base_lrs = [group["lr"] for group in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        for i, base_lr in enumerate(self.base_lrs):
            if self.current_step < self.warmup_steps:
                lr = base_lr * self.current_step / self.warmup_steps
            else:
                progress = (self.current_step - self.warmup_steps) / (
                    self.total_steps - self.warmup_steps
                )
                lr = self.min_lr + (base_lr - self.min_lr) * 0.5 * (
                    1 + math.cos(math.pi * progress)
                )
            self.optimizer.param_groups[i]["lr"] = lr


steps_per_epoch = len(loaders["train"])
total_steps = steps_per_epoch * CONFIG["max_epochs"]
warmup_steps = int(0.05 * total_steps)

sch_sim = WarmupCosineScheduler(optim_sim, warmup_steps, total_steps)
sch_clip = WarmupCosineScheduler(optim_clip, warmup_steps, total_steps)
sch_det = WarmupCosineScheduler(optim_det, warmup_steps, total_steps)

print(f"✅ Schedulers created")

In [ ]:
# Loss functions
loss_cos = nn.CosineEmbeddingLoss(margin=0.2)
loss_ce = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
loss_kl = nn.KLDivLoss(reduction="batchmean")


def soft_xe(logits, soft_target):
    return -(soft_target * F.log_softmax(logits, 1)).sum() / logits.size(0)


print("✅ Loss functions created")

In [ ]:
# Multi-task run_epoch function
def run_epoch(loader, train=True, epoch_num=None):
    if train:
        model.train()
    else:
        model.eval()

    tot_loss, tot_ok, tot_n = 0.0, 0, 0
    all_y, all_p = [], []
    epoch_sim_loss, epoch_clip_loss, epoch_det_loss = 0.0, 0.0, 0.0
    num_batches = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for text, image, label in tqdm(loader, desc="Train" if train else "Eval"):
            text = text.to(DEVICE)
            image = image.to(DEVICE)
            label = label.to(DEVICE)
            bs = label.size(0)

            # Task 1a: Similarity
            ft, fi_m, fi_u = make_pair_sim(text, image, label)
            ta_m, ia_m, _ = model.similarity_module(ft, fi_m)
            ta_u, ia_u, _ = model.similarity_module(ft, fi_u)
            t_cat = torch.cat([ta_m, ta_u])
            i_cat = torch.cat([ia_m, ia_u])
            y_cos = torch.cat(
                [
                    torch.ones(ta_m.size(0), device=DEVICE),
                    -torch.ones(ta_u.size(0), device=DEVICE),
                ]
            )
            ls = loss_cos(t_cat, i_cat, y_cos)
            epoch_sim_loss += ls.item()

            if train:
                optim_sim.zero_grad()
                ls.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.similarity_module.parameters(), CONFIG["grad_clip"]
                )
                optim_sim.step()
                sch_sim.step()

            # Task 1b: CLIP
            text_pooled = text.mean(dim=2)
            ie, te = model.clip_module(image, text_pooled)
            logits = ie @ te.T * math.exp(0.07)
            ids = torch.arange(bs, device=DEVICE)

            ts, is_, _ = model.similarity_module(text, image)
            soft_m = is_ @ ts.T * math.exp(0.07)

            lc = (loss_ce(logits, ids) + loss_ce(logits.T, ids)) / 2
            ls2 = (
                soft_xe(logits, F.softmax(soft_m, 1))
                + soft_xe(logits.T, F.softmax(soft_m.T, 1))
            ) / 2
            l_clip = lc + 0.2 * ls2
            epoch_clip_loss += l_clip.item()

            if train:
                optim_clip.zero_grad()
                l_clip.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.clip_module.parameters(), CONFIG["grad_clip"]
                )
                optim_clip.step()
                sch_clip.step()

            # Task 2: Detection
            with torch.no_grad() if not train else torch.enable_grad():
                ie2, te2 = model.clip_module(image, text_pooled)
            det, attn, skl = model.detection_module(text, image, te2, ie2)
            ld = loss_ce(det, label) + 0.5 * loss_kl(
                F.log_softmax(attn, 1), F.softmax(skl, 1)
            )
            epoch_det_loss += ld.item()

            if train:
                optim_det.zero_grad()
                ld.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.detection_module.parameters(), CONFIG["grad_clip"]
                )
                optim_det.step()
                sch_det.step()

            pred = det.argmax(1)
            tot_ok += pred.eq(label).sum().item()
            tot_n += bs
            tot_loss += ld.item() * bs
            all_y.extend(label.cpu().numpy())
            all_p.extend(pred.cpu().numpy())
            num_batches += 1

    metrics = {
        "loss": tot_loss / tot_n,
        "acc": tot_ok / tot_n,
        "sim_loss": epoch_sim_loss / num_batches,
        "clip_loss": epoch_clip_loss / num_batches,
        "det_loss": epoch_det_loss / num_batches,
    }

    return metrics, all_y, all_p


print("✅ run_epoch defined with multi-task training")

In [ ]:
# MLflow setup
EXPERIMENT_NAME = "vietnamese-coolant-training"
mlflow.set_experiment(EXPERIMENT_NAME)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"coolant-{timestamp}"
mlflow.start_run(run_name=run_name)

mlflow.log_params(CONFIG)
print(f"✅ MLflow run started: {run_name}")

In [ ]:
# Training loop with early stopping
best_val_loss = float("inf")
best_val_acc = 0.0
patience_counter = 0
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

print(f"🚀 Starting training for max {CONFIG['max_epochs']} epochs...")
print(f"   Early stopping patience: {CONFIG['patience']}\n")

for epoch in range(1, CONFIG["max_epochs"] + 1):
    train_metrics, _, _ = run_epoch(loaders["train"], train=True, epoch_num=epoch)
    val_metrics, _, _ = run_epoch(loaders["dev"], train=False, epoch_num=epoch)

    history["train_loss"].append(train_metrics["loss"])
    history["train_acc"].append(train_metrics["acc"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["acc"])

    mlflow.log_metrics(
        {
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "lr_sim": optim_sim.param_groups[0]["lr"],
        },
        step=epoch,
    )

    print(f"Epoch [{epoch:02d}/{CONFIG['max_epochs']}] ")
    print(f"  Train: loss={train_metrics['loss']:.4f} acc={train_metrics['acc']:.4f}")
    print(f"  Val:   loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f}")

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_val_acc = val_metrics["acc"]
        patience_counter = 0

        checkpoint_path = SAVE_DIR / "best_model.pth"
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optim_sim_state_dict": optim_sim.state_dict(),
                "optim_clip_state_dict": optim_clip.state_dict(),
                "optim_det_state_dict": optim_det.state_dict(),
                "val_loss": best_val_loss,
                "val_acc": best_val_acc,
                "config": CONFIG,
            },
            checkpoint_path,
        )
        mlflow.log_artifact(str(checkpoint_path), artifact_path="checkpoints")
        print(f"  ✓ New best val_loss={best_val_loss:.4f}, saved checkpoint")
    else:
        patience_counter += 1
        print(f"  → No improvement ({patience_counter}/{CONFIG['patience']})")

    if patience_counter >= CONFIG["patience"]:
        print(f"\n🛑 Early stopping triggered at epoch {epoch}")
        break
    print()

print(f"\n✅ Training complete!")
print(f"   Best val_loss: {best_val_loss:.4f}")
print(f"   Best val_acc: {best_val_acc:.4f}")

In [ ]:
# Load best model
checkpoint = torch.load(SAVE_DIR / "best_model.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']}")

In [ ]:
# Test evaluation
print("\n🔍 Evaluating on test set...")
test_metrics, test_y, test_p = run_epoch(loaders["test"], train=False)

print(f"\n✅ Test Results:")
print(f"   Loss: {test_metrics['loss']:.4f}")
print(f"   Accuracy: {test_metrics['acc']:.4f}")

report = classification_report(test_y, test_p, target_names=["Real", "Fake"], digits=4)
print("\nClassification Report:")
print(report)

cm = confusion_matrix(test_y, test_p)
print("\nConfusion Matrix:")
print(cm)

mlflow.log_metrics({"test_loss": test_metrics["loss"], "test_acc": test_metrics["acc"]})

report_path = SAVE_DIR / "test_report.txt"
with open(report_path, "w") as f:
    f.write("Test Classification Report\n")
    f.write("=" * 50 + "\n\n")
    f.write(report)
    f.write("\n\nConfusion Matrix:\n")
    f.write(str(cm))

mlflow.log_artifact(str(report_path), artifact_path="results")
print("\n✅ Test report saved")

In [ ]:
# End MLflow run
mlflow.end_run()
print("✅ MLflow run ended")